# Ingeniería del Siglo XXI: Sistema de Frenado Autónomo 🚗

Los automóviles modernos, como los de Tesla o Volvo, utilizan sistemas de "Control de Crucero Adaptativo" para mantener una distancia segura con el vehículo de adelante. No se limitan a "avanzar y frenar" bruscamente; en su lugar, calculan suavemente la aceleración necesaria utilizando matemáticas avanzadas. 

En este laboratorio, dejaremos atrás las secuencias rígidas y programaremos un **Controlador Proporcional**, el algoritmo matemático fundamental de la robótica industrial y la automatización.


In [4]:
import random
from IPython.display import display, HTML

# UID para evitar colisiones en Jupyter Book
uid = str(random.randint(10000, 99999))

simulacion_html = """
<div style="border: 1px solid #e0e0e0; padding: 20px; border-radius: 8px; background-color: #f8f9fa; font-family: -apple-system, sans-serif;">
    <h3 style="margin-top:0; color: #2c3e50;">Simulador: Control Proporcional (Frenado Autónomo)</h3>
    <p style="font-size: 0.95em; color: #555; margin-bottom: 15px;">
        Mueve el obstáculo y ajusta la <strong>Constante Proporcional (Kp)</strong>. Enciende el sonido para "escuchar" el motor del vehículo. Si el Kp es muy alto, escucharás cómo la inyección de aceleración se vuelve inestable y oscila violentamente.
    </p>
    
    <div style="display: flex; gap: 20px; margin-bottom: 20px; background: #e9ecef; padding: 15px; border-radius: 8px; flex-wrap: wrap; align-items: center; justify-content: space-between;">
        <div style="display: flex; gap: 10px; align-items: center; flex-wrap: wrap;">
            <button id="power_UID" style="background-color: #198754; color: white; border: none; padding: 10px 15px; border-radius: 5px; cursor: pointer; font-weight: bold; font-size: 14px;">▶ Encender Motor</button>
            <button id="reset_UID" style="background-color: #6c757d; color: white; border: none; padding: 10px 15px; border-radius: 5px; cursor: pointer; font-weight: bold; font-size: 14px;">🔄 Reiniciar</button>
            
            <label style="cursor: pointer; display: flex; align-items: center; gap: 5px; background: #fff; padding: 8px 12px; border-radius: 5px; border: 1px solid #ccc; font-size: 14px; font-weight: bold; color: #495057; margin-left: 10px;">
                <input type="checkbox" id="sound_chk_UID" style="cursor: pointer; width: 16px; height: 16px;">
                🔊 Sonido Motor
            </label>
        </div>
        
        <div style="display: flex; gap: 20px; align-items: center;">
            <div style="display: flex; flex-direction: column; align-items: center;">
                <label style="font-weight: 600; font-size: 0.8em; color: #495057;">Posición del Obstáculo</label>
                <input type="range" id="target_pos_UID" min="25" max="100" value="60" style="width: 140px; cursor: pointer;">
            </div>
            
            <div style="display: flex; flex-direction: column; align-items: center;">
                <label style="font-weight: 600; font-size: 0.8em; color: #6f42c1;">Constante Kp</label>
                <div style="display: flex; align-items: center; gap: 5px;">
                    <input type="range" id="kp_val_UID" min="0.5" max="15" step="0.5" value="3" style="width: 100px; cursor: pointer;">
                    <span id="lbl_kp_UID" style="font-weight: bold; font-family: monospace; width: 30px;">3.0</span>
                </div>
            </div>
        </div>
    </div>

    <!-- Panel de Telemetría -->
    <div style="display: flex; gap: 15px; justify-content: center; margin-bottom: 15px; flex-wrap: wrap;">
        <div style="background: #fff; padding: 8px 15px; border-radius: 5px; border: 1px solid #ccc; width: 120px; text-align: center;">
            <label style="font-weight: 600; font-size: 0.8em; margin-bottom: 2px; color: #0d6efd; display: block;">Distancia</label>
            <span id="tele_dist_UID" style="font-weight: bold; font-family: monospace; font-size: 1.1em; color: #212529;">40.0 cm</span>
        </div>
        <div style="background: #fff; padding: 8px 15px; border-radius: 5px; border: 1px solid #ccc; width: 120px; text-align: center;">
            <label style="font-weight: 600; font-size: 0.8em; margin-bottom: 2px; color: #dc3545; display: block;">Error (d - 20)</label>
            <span id="tele_err_UID" style="font-weight: bold; font-family: monospace; font-size: 1.1em; color: #212529;">20.0 cm</span>
        </div>
        <div style="background: #fff; padding: 8px 15px; border-radius: 5px; border: 1px solid #ccc; width: 120px; text-align: center;">
            <label style="font-weight: 600; font-size: 0.8em; margin-bottom: 2px; color: #fd7e14; display: block;">Potencia Motor</label>
            <span id="tele_pot_UID" style="font-weight: bold; font-family: monospace; font-size: 1.1em; color: #212529;">0 %</span>
        </div>
    </div>
    
    <div style="display: flex; justify-content: center; margin-bottom: 10px;">
        <canvas id="canvas_sim_UID" width="700" height="200" style="background: #ffffff; border: 1px solid #ccc; border-radius: 4px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); max-width: 100%;"></canvas>
    </div>
</div>

<script>
    (function() {
        function initSimulation() {
            const canvas = document.getElementById('canvas_sim_UID');
            if (!canvas) {
                setTimeout(initSimulation, 50); 
                return;
            }
            
            const ctx = canvas.getContext('2d');
            const btn_power = document.getElementById('power_UID');
            const btn_reset = document.getElementById('reset_UID');
            
            // Inputs
            const slider_target = document.getElementById('target_pos_UID');
            const slider_kp = document.getElementById('kp_val_UID');
            const lbl_kp = document.getElementById('lbl_kp_UID');
            const chk_sound = document.getElementById('sound_chk_UID');
            
            // Telemetría
            const tele_dist = document.getElementById('tele_dist_UID');
            const tele_err = document.getElementById('tele_err_UID');
            const tele_pot = document.getElementById('tele_pot_UID');
            
            // Constantes Físicas y de Escala
            const PX_PER_CM = 6;
            const Y_FLOOR = 150;
            const OBJETIVO_CM = 20;
            const MAX_SPEED_CM_S = 35; 
            const START_X = 50;
            
            // Estado del Sistema
            let isPowerOn = false;
            let lastTime = 0;
            let robotX = START_X;
            let targetCm = parseFloat(slider_target.value);
            let Kp = parseFloat(slider_kp.value);
            
            // Variables de cálculo
            let potencia = 0;
            let currentDistCm = 0;
            let error = 0;

            // Motor de Audio Avanzado (Síntesis de Automóvil)
            let audioCtx = null;
            let osc1 = null;      // Rugido principal
            let osc2 = null;      // Vibración de cuerpo (Sub)
            let filter = null;    // Emulación del tubo de escape
            let masterGain = null;

            function initAudio() {
                if (!audioCtx) {
                    audioCtx = new (window.AudioContext || window.webkitAudioContext)();
                    
                    osc1 = audioCtx.createOscillator();
                    osc2 = audioCtx.createOscillator();
                    filter = audioCtx.createBiquadFilter();
                    masterGain = audioCtx.createGain();
                    
                    // Configuramos las ondas para que suenen mecánicas y ásperas
                    osc1.type = 'sawtooth';
                    osc2.type = 'square';
                    
                    // El filtro simula el escape: corta los agudos artificiales
                    filter.type = 'lowpass';
                    filter.Q.value = 2; // Ligera resonancia
                    
                    masterGain.gain.value = 0;
                    
                    // Conexiones: Osciladores -> Filtro -> Volumen -> Salida
                    osc1.connect(filter);
                    osc2.connect(filter);
                    filter.connect(masterGain);
                    masterGain.connect(audioCtx.destination);
                    
                    osc1.start();
                    osc2.start();
                }
                if (audioCtx.state === 'suspended') {
                    audioCtx.resume();
                }
            }

            function updateAudioMotor(potencia_actual) {
                if (!audioCtx || !osc1 || !masterGain) return;
                
                if (chk_sound.checked && isPowerOn && Math.abs(potencia_actual) > 1) {
                    let absPot = Math.abs(potencia_actual);
                    
                    // Matemáticas del Motor:
                    // Frecuencia base (RPM): Ralentí ~30Hz, Aceleración a fondo ~150Hz
                    let freq = 30 + (absPot * 1.5); 
                    
                    osc1.frequency.setTargetAtTime(freq, audioCtx.currentTime, 0.05);
                    // El segundo oscilador va una octava más abajo (frecuencia a la mitad) para darle profundidad
                    osc2.frequency.setTargetAtTime(freq * 0.5, audioCtx.currentTime, 0.05);
                    
                    // El filtro de escape se abre al acelerar, dejando salir el rugido
                    let filterFreq = 150 + (absPot * 15);
                    filter.frequency.setTargetAtTime(filterFreq, audioCtx.currentTime, 0.05);
                    
                    // El volumen aumenta con la potencia
                    let vol = 0.1 + (absPot / 100) * 0.2;
                    masterGain.gain.setTargetAtTime(vol, audioCtx.currentTime, 0.05);
                } else {
                    // Silenciar suavemente si está apagado o frenado
                    masterGain.gain.setTargetAtTime(0, audioCtx.currentTime, 0.1);
                }
            }

            function drawRobot(x, y) {
                const w = 60; 
                ctx.fillStyle = "#333";
                ctx.fillRect(x + 5, y - 10, 15, 20);
                ctx.fillRect(x + 40, y - 10, 15, 20);
                
                ctx.fillStyle = "#f8f9fa";
                ctx.strokeStyle = "#adb5bd";
                ctx.lineWidth = 2;
                ctx.beginPath();
                ctx.roundRect(x, y - 40, w, 35, 5);
                ctx.fill(); ctx.stroke();
                
                ctx.fillStyle = "#ffc107";
                ctx.beginPath();
                ctx.arc(x + w, y - 20, 8, -Math.PI/2, Math.PI/2);
                ctx.fill();
                ctx.fillStyle = "#212529";
                ctx.beginPath();
                ctx.arc(x + w, y - 25, 3, 0, Math.PI*2);
                ctx.arc(x + w, y - 15, 3, 0, Math.PI*2);
                ctx.fill();
                
                ctx.strokeStyle = "rgba(40, 167, 69, 0.5)";
                ctx.lineWidth = 4;
                ctx.beginPath();
                ctx.moveTo(x + w, y - 5);
                ctx.lineTo(x + w + (OBJETIVO_CM * PX_PER_CM), y - 5);
                ctx.stroke();
                
                ctx.beginPath();
                ctx.moveTo(x + w + (OBJETIVO_CM * PX_PER_CM), y - 15);
                ctx.lineTo(x + w + (OBJETIVO_CM * PX_PER_CM), y + 5);
                ctx.stroke();
                
                ctx.fillStyle = "#28a745";
                ctx.font = "10px sans-serif";
                ctx.fillText("Objetivo (20cm)", x + w + 10, y + 10);
            }

            function drawTarget(x_cm) {
                let px = 50 + (x_cm * PX_PER_CM);
                
                ctx.fillStyle = "#6c757d";
                ctx.fillRect(px, Y_FLOOR - 50, 40, 50);
                ctx.strokeStyle = "#495057";
                ctx.lineWidth = 2;
                ctx.strokeRect(px, Y_FLOOR - 50, 40, 50);
                
                ctx.fillStyle = "#fff";
                ctx.font = "bold 12px sans-serif";
                ctx.fillText("OBJ", px + 8, Y_FLOOR - 20);
                
                return px;
            }

            function drawSim() {
                ctx.clearRect(0, 0, canvas.width, canvas.height);
                
                ctx.fillStyle = "#e9ecef";
                ctx.fillRect(0, Y_FLOOR, canvas.width, canvas.height - Y_FLOOR);
                ctx.strokeStyle = "#ced4da";
                ctx.lineWidth = 1;
                ctx.beginPath(); ctx.moveTo(0, Y_FLOOR); ctx.lineTo(canvas.width, Y_FLOOR); ctx.stroke();
                
                ctx.fillStyle = "#adb5bd";
                ctx.font = "10px sans-serif";
                for(let i=0; i<=100; i+=10) {
                    let markPx = 50 + (i * PX_PER_CM);
                    ctx.beginPath(); ctx.moveTo(markPx, Y_FLOOR); ctx.lineTo(markPx, Y_FLOOR + 5); ctx.stroke();
                    if(i % 20 === 0) ctx.fillText(i + "cm", markPx - 10, Y_FLOOR + 15);
                }

                targetCm = parseFloat(slider_target.value);
                Kp = parseFloat(slider_kp.value);
                lbl_kp.innerText = Kp.toFixed(1);
                
                let targetPx = drawTarget(targetCm);
                drawRobot(robotX, Y_FLOOR);
                
                let frontRobot = robotX + 60;
                let distPx = targetPx - frontRobot;
                currentDistCm = distPx / PX_PER_CM;
                
                if (currentDistCm < 0) {
                    currentDistCm = 0;
                    robotX = targetPx - 60; 
                }
                
                if (isPowerOn) {
                    error = currentDistCm - OBJETIVO_CM;
                    potencia = error * Kp;
                    
                    if (potencia > 100) potencia = 100;
                    if (potencia < -100) potencia = -100;
                } else {
                    potencia = 0;
                    error = currentDistCm - OBJETIVO_CM;
                }
                
                tele_dist.innerText = currentDistCm.toFixed(1) + " cm";
                tele_err.innerText = error.toFixed(1) + " cm";
                tele_pot.innerText = Math.round(potencia) + " %";
                
                tele_err.style.color = Math.abs(error) < 0.5 ? "#28a745" : "#dc3545";
                tele_pot.style.color = potencia === 0 ? "#212529" : (potencia > 0 ? "#0d6efd" : "#fd7e14");
                
                if (Math.abs(potencia) > 5) {
                    let cx = robotX + 30;
                    let cy = Y_FLOOR - 55;
                    let len = (potencia / 100) * 40;
                    
                    ctx.strokeStyle = potencia > 0 ? "#0d6efd" : "#fd7e14";
                    ctx.fillStyle = ctx.strokeStyle;
                    ctx.lineWidth = 3;
                    
                    ctx.beginPath();
                    ctx.moveTo(cx, cy);
                    ctx.lineTo(cx + len, cy);
                    ctx.stroke();
                    
                    ctx.beginPath();
                    if (potencia > 0) {
                        ctx.moveTo(cx + len, cy);
                        ctx.lineTo(cx + len - 6, cy - 4);
                        ctx.lineTo(cx + len - 6, cy + 4);
                    } else {
                        ctx.moveTo(cx + len, cy);
                        ctx.lineTo(cx + len + 6, cy - 4);
                        ctx.lineTo(cx + len + 6, cy + 4);
                    }
                    ctx.fill();
                }
            }

            function animate(timestamp) {
                if (!lastTime) lastTime = timestamp;
                let deltaTime = (timestamp - lastTime) / 1000;
                lastTime = timestamp;
                
                if (deltaTime > 0.1) deltaTime = 0.1; 
                
                if (isPowerOn) {
                    let speedCmS = (potencia / 100) * MAX_SPEED_CM_S;
                    let speedPxS = speedCmS * PX_PER_CM;
                    robotX += speedPxS * deltaTime;
                    
                    if (robotX < 0) robotX = 0;
                }
                
                updateAudioMotor(potencia);
                
                drawSim();
                requestAnimationFrame(animate);
            }

            btn_power.addEventListener('click', () => {
                initAudio();
                isPowerOn = !isPowerOn;
                if (isPowerOn) {
                    btn_power.innerText = "⏹ Apagar Motor";
                    btn_power.style.backgroundColor = "#dc3545";
                } else {
                    btn_power.innerText = "▶ Encender Motor";
                    btn_power.style.backgroundColor = "#198754";
                }
            });

            btn_reset.addEventListener('click', () => {
                isPowerOn = false;
                robotX = START_X;
                slider_target.value = 60;
                slider_kp.value = 3;
                
                btn_power.innerText = "▶ Encender Motor";
                btn_power.style.backgroundColor = "#198754";
                
                if (audioCtx && masterGain) {
                    masterGain.gain.setTargetAtTime(0, audioCtx.currentTime, 0.05);
                }
                
                drawSim();
            });

            chk_sound.addEventListener('change', () => {
                if (chk_sound.checked) initAudio();
            });

            slider_target.addEventListener('input', drawSim);
            slider_kp.addEventListener('input', drawSim);
            
            requestAnimationFrame(animate);
        }
        
        initSimulation();
    })();
</script>
"""

html_final = simulacion_html.replace('UID', uid)
display(HTML(html_final))


## 1. Guía del Estudiante

### El Desafío
Tu misión es programar el Ladrillo NXT para que actúe como un vehículo autónomo. El robot debe "perseguir" a un compañero que camina delante de él, manteniendo siempre una **distancia exacta de 20 cm**. 
* Si el compañero se aleja rápido, el robot debe acelerar al máximo.
* Si el compañero camina lento, el robot debe ir lento.
* Si el compañero se detiene a 20 cm, el robot debe detenerse suavemente.
* Si el compañero retrocede y se acerca a menos de 20 cm, el robot debe ir en reversa para recuperar su distancia de seguridad.

### Fase 1: Ingeniería (Construcción)
1. Construye la **Base Motriz Estándar** con los motores conectados a los **puertos B y C**.
2. Monta el **Sensor Ultrasónico** en el frente del chasis.
3. Conecta el sensor ultrasónico al **Puerto 4**.

### Fase 2: Programación y Matemáticas (Código NXC)
Para lograr este comportamiento fluido, no podemos usar simples comandos `if/else`. Necesitamos una **función matemática lineal**. 

Calcularemos el "Error" (la diferencia entre dónde estamos y dónde queremos estar) y multiplicaremos ese error por una Constante Proporcional ($K_p$). 

$$ Potencia = K_p \times (Distancia_{Actual} - Distancia_{Objetivo}) $$

Copia el siguiente código en **BricxCC** y analiza cómo esta fórmula se transforma en programación C:

```c
// Definimos nuestras variables estáticas (Constantes)
#define OBJETIVO 20  // Queremos mantener siempre 20 cm de distancia
#define KP 3         // Constante Proporcional (Multiplicador de potencia)

task main() {
    SetSensorLowspeed(S4);
    
    int distancia;
    int error;
    int potencia;

    while(true) {
        // 1. Leer el sensor
        distancia = SensorUS(S4);

        // El sensor ultrasónico devuelve 255 si no detecta nada. 
        // Filtramos ese valor para que el robot no se vuelva loco.
        if (distancia < 255) {
            
            // 2. Calcular el Error Matemático
            error = distancia - OBJETIVO;
            
            // 3. Calcular la Potencia (Función lineal)
            potencia = error * KP;

            // 4. Limites de seguridad (Los motores del NXT solo aceptan valores entre -100 y 100)
            if (potencia > 100) {
                potencia = 100;
            }
            if (potencia < -100) {
                potencia = -100;
            }

            // 5. Enviar la potencia calculada a los motores
            // Nota: Si la potencia es negativa, OnFwd automáticamente hará girar los motores en reversa.
            OnFwd(OUT_BC, potencia);
            
        } else {
            // Si no detecta a nadie frente a él, se detiene por seguridad.
            Off(OUT_BC);
        }
        
        // Pequeña pausa de 50ms para permitir que el sensor tome una nueva lectura
        Wait(50); 
    }
}
```

### Fase 3: Pruebas y Reflexión
* **Compila y descarga** el programa.
* Pon el robot en el suelo. Coloca tu mano o un cuaderno frente a él a 20 cm. Muévelo lentamente hacia adelante y hacia atrás. El robot debería "imantar" su movimiento al tuyo.
* **Experimento:** En el código, cambia el valor de la constante `KP` de `3` a `10`. Vuelve a compilar. ¿Qué ocurre con el movimiento del robot? ¿Por qué se vuelve inestable o "nervioso"? Anota tus conclusiones.

---

## 2. Guía del Docente

> **Nota para el Profesor:** Este laboratorio es una excelente oportunidad para demostrar la utilidad práctica de las funciones matemáticas que los alumnos estudian en la pizarra. Representa la introducción a la "Teoría de Control", específicamente a la letra "P" de los famosos Controladores PID.

### Vínculo Curricular (MINEDUC Chile)
* **Matemáticas (1º a 3º Medio):** Modelamiento matemático. Uso de funciones lineales ($f(x) = mx + c$) para modelar situaciones de la vida real. Relación de variables proporcionales.
* **Física (2º Medio):** Cinemática. Conceptos de velocidad relativa, sistemas de referencia y aceleración variable.
* **Tecnología (3º y 4º Medio):** Resolución de problemas mediante la creación de algoritmos y programación estructurada.

### Conceptos Clave a Desarrollar
1. **La Variable de Error ($e$):** En robótica, el error no es una equivocación, es un dato. Es la resta entre el *Setpoint* (estado deseado) y el *Process Variable* (estado actual). Si estamos a 30 cm y queremos estar a 20 cm, el error es +10. Si estamos a 15 cm, el error es -5.
2. **Control Proporcional ($K_p$):** Enseña cómo la pendiente de la función afecta al sistema. 
    * Si $K_p$ es muy bajo (ej: 1), el robot reaccionará con lentitud matemática (Error 10 * 1 = 10% de potencia). 
    * Si $K_p$ es muy alto (ej: 15), el robot será agresivo y oscilará violentamente tratando de corregir su posición, sobrepasando el objetivo repetidamente (Efecto de sobreoscilación u *Overshoot*).
3. **Manejo de Signos:** Destaque cómo el lenguaje NXC interpreta la potencia negativa. Un valor de potencia de -30 en el comando `OnFwd` invierte automáticamente la polaridad de los motores, logrando la reversa sin necesidad de usar bloques `if/else` separados para avanzar o retroceder.

### Resolución de Problemas (Troubleshooting)
* **Comportamiento errático aleatorio:** El sensor ultrasónico del NXT a veces arroja lecturas falsas rebotando en el suelo (generalmente retorna el valor 255). La estructura `if (distancia < 255)` está ahí específicamente para filtrar ese "ruido" de los datos.
* **El robot gira sobre su eje:** Revise que los motores estén conectados a los puertos B y C, y que ningún cable esté suelto. Si un motor recibe potencia y el otro no, el robot pivotará en lugar de avanzar recto.
* **Oscilación extrema (El robot tiembla violentamente al llegar a los 20 cm):** Esto significa que la constante `KP` está demasiado alta para la superficie de trabajo. Pida a los alumnos que la reduzcan gradualmente (de 3 a 2, o a 1.5).